In [2]:
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from scipy import sparse

In [5]:
svm_model = joblib.load("svm_model.pkl")

X_scraped_tfidf = joblib.load("../feature_engineering/scraped_tfidf_matrix.pkl")  # sparse matrix
X_scraped_struct = joblib.load("../feature_engineering/scraped_feature_list.pkl") # DataFrame 

tfidf = joblib.load("../feature_engineering/tfidf_vectorizer.pkl")
svd = joblib.load("../XGboost_model_folder/tfidf_svd.pkl")

#need to transform with same svd as tested on
X_scraped_tfidf_svd = svd.transform(X_scraped_tfidf)

def to_numeric_sparse(df):
    df = df.copy()
    bool_cols = df.select_dtypes(include="bool").columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(np.int8)
    return sparse.csr_matrix(df.values)

X_scraped_struct_sparse = to_numeric_sparse(X_scraped_struct)

# Convert SVD output to sparse (optional)
X_scraped_tfidf_sparse = sparse.csr_matrix(X_scraped_tfidf_svd)

# Combine SVD + structured features
X_scraped_combined = sparse.hstack([X_scraped_tfidf_sparse, X_scraped_struct_sparse])

In [8]:
y_pred = svm_model.predict(X_scraped_combined)
np.unique(y_pred, return_counts=True)

(array([0, 1]), array([8177,   50]))

In [10]:
fraud_indices = [i for i, pred in enumerate(y_pred) if pred == 1]
fraud_jobs = X_scraped_struct.iloc[fraud_indices]
fraud_jobs.head()

,salary_avg_norm,location_legitimacy,loc_len,loc_word_count,has_us_prefix,has_state_code,starts_with_direction,contains_digits,industry_group_Education,industry_group_Engineering/Construction,industry_group_Finance,industry_group_Government/Nonprofit,industry_group_Healthcare,industry_group_Manufacturing/Industrial,industry_group_Other,industry_group_Other Business/Services,industry_group_Retail/Hospitality,industry_group_Technology,industry_group_Transportation
73,9.387239e-04,5,-48,-88,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0
615,4.776999e-04,5,-124,18,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
808,1.012167e-03,1,12,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0
892,9.331694e-04,1,3,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0
1054,5.863090e-07,3,2,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0


In [11]:
scores = svm_model.decision_function(X_scraped_combined)
X_scraped_struct["fraud_pred"] = y_pred
X_scraped_struct["fraud_score"] = scores

top_suspicious = X_scraped_struct[X_scraped_struct.fraud_pred == 1].sort_values("fraud_score", ascending=False)
top_suspicious.head(10)


,salary_avg_norm,location_legitimacy,loc_len,loc_word_count,has_us_prefix,has_state_code,starts_with_direction,contains_digits,industry_group_Education,industry_group_Engineering/Construction,...,industry_group_Government/Nonprofit,industry_group_Healthcare,industry_group_Manufacturing/Industrial,industry_group_Other,industry_group_Other Business/Services,industry_group_Retail/Hospitality,industry_group_Technology,industry_group_Transportation,fraud_pred,fraud_score
6929,0.000544,5,-33,-100,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,1,3.744773
7294,0.000278,3,-69,-71,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,3.427601
7706,0.000555,3,-58,-70,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,3.214669
4102,0.000000,5,-9,-98,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2.979069
6302,0.000731,3,21,-98,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,2.918278
73,0.000939,5,-48,-88,0,1,0,1,1,0,...,0,0,0,0,0,0,0,0,1,2.672574
6810,0.000411,5,67,-85,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,1,2.176246
4806,0.000444,5,-58,-60,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1.887708
6717,0.000767,3,-49,-22,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,1,1.662207
4235,0.000422,5,-108,-19,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0.798901


In [14]:
vectorizer = joblib.load("../feature_engineering/tfidf_vectorizer.pkl")

In [15]:
coef = svm_model.coef_[0]   # no .toarray()
feature_names = vectorizer.get_feature_names_out()

# Top fraud indicators
top_fraud = coef.argsort()[-20:]
[(feature_names[i], coef[i]) for i in reversed(top_fraud)]


[('00', np.float64(7.919419496782018)),
 ('ability think', np.float64(4.038604566245324)),
 ('accordingly', np.float64(3.242964872223225)),
 ('20 years', np.float64(3.0705301062676615)),
 ('ability understand', np.float64(3.0045062449471143)),
 ('2000', np.float64(2.6364557375011253)),
 ('13', np.float64(2.598456328010216)),
 ('2018', np.float64(2.4691957629077494)),
 ('1981', np.float64(2.3620128217051124)),
 ('75', np.float64(2.3456891268169664)),
 ('1995', np.float64(2.329895369844761)),
 ('ability lift', np.float64(2.3104529608281155)),
 ('361', np.float64(2.291509486312127)),
 ('acceptable', np.float64(2.2907466469386675)),
 ('17', np.float64(2.2801544066863646)),
 ('1987', np.float64(2.239414930967869)),
 ('200 people', np.float64(2.2032773273360085)),
 ('500', np.float64(2.188477801101627)),
 ('2017', np.float64(2.171026423125724)),
 ('accommodationswillingness', np.float64(2.01759121418456))]

In [16]:
coef = svm_model.coef_[0]
feature_names = vectorizer.get_feature_names_out()

top_fraud = coef.argsort()[-20:]
for i in reversed(top_fraud):
    print(feature_names[i], coef[i])


00 7.919419496782018
ability think 4.038604566245324
accordingly 3.242964872223225
20 years 3.0705301062676615
ability understand 3.0045062449471143
2000 2.6364557375011253
13 2.598456328010216
2018 2.4691957629077494
1981 2.3620128217051124
75 2.3456891268169664
1995 2.329895369844761
ability lift 2.3104529608281155
361 2.291509486312127
acceptable 2.2907466469386675
17 2.2801544066863646
1987 2.239414930967869
200 people 2.2032773273360085
500 2.188477801101627
2017 2.171026423125724
accommodationswillingness 2.01759121418456
